# Semana 14 — Hybrid Search e Multi-Stage Reranking
**Fase 3: Advanced RAG, Search e Reranking** · Gabarito (notebook resolvido)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/ai-architect-foundations/blob/main/03-Phase-3-Advanced-RAG/notebooks/semana_14_gabarito.ipynb)

Leitura de conceito desta semana: [Semana-14-Conceitos.md](https://github.com/NetoRibeiro/ai-architect-foundations/blob/main/03-Phase-3-Advanced-RAG/docs/Semana-14-Conceitos.md)

Use este gabarito **depois** de tentar o notebook de exercícios. Compare a sua solução, não copie.

> **GPU:** em *Ambiente de execução → Alterar o tipo de ambiente de execução*, selecione **T4 GPU** antes de começar.

In [ ]:
!pip -q install sentence-transformers rank_bm25

In [ ]:
import numpy as np, re, time, pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
MODELO_EMB = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MODELO_RERANK = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
# --- Corpus de demonstração: base de conhecimento fictícia de uma empresa ---
import random
def corpus_demo(seed=7):
    """Devolve (docs, consultas).
    docs: lista de {id, texto, area, ano}.
    consultas: lista de {q, resposta, relevantes: {doc_id: grau}}; grau 2 = contém a resposta, 1 = mesmo assunto."""
    rnd = random.Random(seed)
    fatos = {
        "ferias": [("O colaborador tem direito a 30 dias de férias após 12 meses de trabalho.", "Quantos dias de descanso anual eu tenho?"),
                   ("As férias podem ser divididas em até três períodos, um deles com no mínimo 14 dias.", "Posso fracionar meu período de descanso?"),
                   ("O pedido de férias deve ser feito no portal RH com 45 dias de antecedência.", "Com que antecedência solicito minha folga anual?")],
        "reembolso": [("Despesas de viagem são reembolsadas em até 10 dias úteis após o envio dos recibos.", "Em quanto tempo recebo de volta o que gastei viajando?"),
                      ("O limite diário de alimentação em viagens nacionais é de R$ 120.", "Qual o teto de gasto com refeição em deslocamento a trabalho?"),
                      ("Corridas de aplicativo só são reembolsadas entre 22h e 6h ou em deslocamentos a clientes.", "A empresa paga meu Uber?")],
        "seguranca": [("Senhas corporativas expiram a cada 90 dias e exigem autenticação em dois fatores.", "De quanto em quanto tempo preciso trocar minha credencial?"),
                      ("Dados de clientes classificados como confidenciais não podem ser enviados a ferramentas externas de IA.", "Posso colar informações de cliente em um chatbot público?"),
                      ("Incidentes de segurança devem ser reportados ao SOC em até 2 horas.", "Qual o prazo para avisar sobre um vazamento?")],
        "ti": [("Notebooks são substituídos a cada 36 meses mediante chamado no service desk.", "Quando posso trocar meu computador?"),
               ("O acesso à VPN exige o cliente GlobalConnect e certificado pessoal.", "Como me conecto à rede da empresa de casa?"),
               ("Licenças de software adicionais precisam de aprovação do gestor e do time de compras.", "Como consigo um programa novo?")],
        "beneficios": [("O plano de saúde cobre dependentes legais sem custo adicional até 21 anos.", "Meu filho pode entrar no convênio médico?"),
                       ("O vale-refeição é creditado no último dia útil de cada mês.", "Em que dia cai o cartão de alimentação?"),
                       ("A empresa subsidia 50% de cursos de idiomas, limitado a R$ 400 por mês.", "Existe ajuda para estudar inglês?")],
    }
    codigos = [("O código de erro VPN-4031 indica certificado pessoal expirado; renove pelo portal de identidades.", "ti", "O que significa o erro VPN-4031?"),
               ("O formulário RH-207 é usado para solicitar licença não remunerada.", "ferias", "Para que serve o formulário RH-207?"),
               ("O centro de custo CC-8812 deve ser usado para despesas de treinamento externo.", "reembolso", "Qual centro de custo uso: CC-8812 é para quê?"),
               ("A política SEG-014 define a classificação de dados em público, interno e confidencial.", "seguranca", "O que diz a política SEG-014?")]
    projetos = ["Atlas", "Boreal", "Cometa", "Delta", "Éden"]
    armadilhas = ["O time do projeto {p} volta de férias em janeiro e retoma as entregas do trimestre.",
                  "No projeto {p}, o reembolso ao cliente foi aprovado após a revisão do contrato.",
                  "A equipe do projeto {p} trocou a senha do ambiente de testes durante a migração.",
                  "O projeto {p} comprou notebooks novos para o laboratório de dados.",
                  "O plano de saúde financeira do projeto {p} prevê corte de 10% nos custos de nuvem."]
    ruido = ["Este procedimento foi revisado pelo comitê interno.", "Dúvidas devem ser encaminhadas ao gestor imediato.",
             "A política vale para todas as unidades do grupo.", "Exceções exigem aprovação formal da diretoria."]
    docs, consultas = [], []
    def novo(texto, area):
        docs.append({"id": len(docs), "texto": texto, "area": area, "ano": rnd.choice([2023, 2024, 2025])}); return docs[-1]["id"]
    por_area = {}
    for area, itens in fatos.items():
        for fato, pergunta in itens:
            i = novo(f"{fato} {rnd.choice(ruido)}", area); por_area.setdefault(area, []).append(i)
            consultas.append({"q": pergunta, "resposta": fato, "relevantes": {i: 2}, "area": area})
    for texto, area, pergunta in codigos:
        i = novo(texto, area); por_area[area].append(i)
        consultas.append({"q": pergunta, "resposta": texto, "relevantes": {i: 2}, "area": area})
    for c in consultas:
        for j in por_area[c["area"]]:
            c["relevantes"].setdefault(j, 1)
    for p in projetos:
        for a in armadilhas:
            novo(a.format(p=p), "projetos")
    return docs, consultas

def manual_demo(docs):
    """Junta os documentos de política em um manual longo em Markdown (para exercícios de chunking)."""
    titulos = {"ferias": "Férias e licenças", "reembolso": "Viagens e reembolsos", "seguranca": "Segurança da informação",
               "ti": "Equipamentos e acesso", "beneficios": "Benefícios"}
    partes = ["# Manual do Colaborador\n\nEste manual reúne as políticas internas vigentes."]
    for area, titulo in titulos.items():
        frases = [d["texto"] for d in docs if d["area"] == area]
        partes.append(f"## {titulo}\n\n" + "\n\n".join(" ".join(frases[i:i + 2]) for i in range(0, len(frases), 2)))
    return "\n\n".join(partes)

docs, consultas = corpus_demo()
textos = [d["texto"] for d in docs]
parafrases = [c for c in consultas if not re.search(r"[A-Z]{2,3}-\d+", c["q"])]
codigos = [c for c in consultas if re.search(r"[A-Z]{2,3}-\d+", c["q"])]
print(len(docs), "docs |", len(parafrases), "consultas em paráfrase |", len(codigos), "consultas com código")

## Passo 1 — Lexical retriever: BM25
Tokenize em minúsculas, construa o index e devolva os índices dos documentos ordenados por score.

In [ ]:
def tokenizar(t): return re.findall(r"\w+", t.lower())

bm25 = BM25Okapi([tokenizar(t) for t in textos])

def rank_bm25(q, k=20):
    scores = bm25.get_scores(tokenizar(q))
    return list(np.argsort(-scores)[:k])

print([textos[i][:60] for i in rank_bm25("O que significa o erro VPN-4031?", 2)])

## Passo 2 — Dense retriever

In [ ]:
emb_model = SentenceTransformer(MODELO_EMB)
embed = lambda ts: emb_model.encode(list(ts), normalize_embeddings=True, convert_to_numpy=True)
E = embed(textos)

def rank_denso(q, k=20):
    return list(np.argsort(-(E @ embed([q])[0]))[:k])

print([textos[i][:60] for i in rank_denso("A empresa paga meu Uber?", 2)])

## Passo 3 — Reciprocal Rank Fusion (RRF)
`listas` é uma lista de rankings (cada um, uma lista de ids em ordem). Some `1 / (k_rrf + posição)` para cada documento, com posição começando em 1.

In [ ]:
def rrf(listas, k_rrf=60, k=20):
    pontos = {}
    for lista in listas:
        for pos, doc in enumerate(lista, start=1):
            pontos[doc] = pontos.get(doc, 0.0) + 1.0 / (k_rrf + pos)
    return sorted(pontos, key=pontos.get, reverse=True)[:k]

def rank_hibrido(q, k=20):
    return rrf([rank_bm25(q), rank_denso(q)], k=k)

assert rrf([[1, 2, 3], [3, 1, 2]])[0] == 1

## Passo 4 — Reranking com cross-encoder
Recupere `n_candidatos` com o híbrido, pontue cada par `(consulta, documento)` com o cross-encoder e reordene.

In [ ]:
reranker = CrossEncoder(MODELO_RERANK)

def rank_rerank(q, k=20, n_candidatos=15):
    cand = rank_hibrido(q, n_candidatos)
    scores = reranker.predict([(q, textos[i]) for i in cand])
    ordem = np.argsort(-np.asarray(scores).reshape(-1))
    return [cand[i] for i in ordem][:k]

## Passo 5 — Quem ganha onde?
MRR (posição recíproca do primeiro documento de grau 2), separado por tipo de query.

In [ ]:
def mrr(fn, conjunto, k=10):
    rr = []
    for c in conjunto:
        ranking = fn(c["q"], k)
        pos = next((p for p, d in enumerate(ranking, start=1) if c["relevantes"].get(int(d), 0) == 2), None)
        rr.append(1.0 / pos if pos else 0.0)
    return float(np.mean(rr))

sistemas = {"BM25": rank_bm25, "Denso": rank_denso, "Híbrido (RRF)": rank_hibrido, "Híbrido + rerank": rank_rerank}
tabela = pd.DataFrame({nome: {"paráfrases": mrr(fn, parafrases), "códigos": mrr(fn, codigos), "todas": mrr(fn, consultas)}
                       for nome, fn in sistemas.items()}).T.round(3)
tabela

Interprete: onde o BM25 vai mal? Onde o denso vai mal? O híbrido é o melhor em tudo ou o mais **robusto**?

## Passo 6 — Latência por stage

In [ ]:
def cronometro(fn, n=20):
    q = consultas[0]["q"]; fn(q)
    t0 = time.perf_counter()
    for _ in range(n): fn(q)
    return (time.perf_counter() - t0) / n * 1000

lat = {nome: cronometro(fn) for nome, fn in sistemas.items()}
tabela["latência (ms)"] = pd.Series(lat).round(1); tabela

## Passo 7 — Quantos candidatos mandar ao reranker?

In [ ]:
for n in (3, 5, 10, 20, 40):
    fn = lambda q, k=10, n=n: rank_rerank(q, k, n_candidatos=n)
    print(f"candidatos={n:3d} | MRR={mrr(fn, consultas):.3f} | {cronometro(fn, 5):6.1f} ms")

## Passo 8 — Query understanding: roteamento por área
Classifique a query em uma área comparando seu embedding com o de uma descrição curta de cada área. Use a área prevista como filtro antes de ranquear.

In [ ]:
descricoes = {"ferias": "férias, folgas, licenças e descanso do colaborador",
              "reembolso": "viagens a trabalho, despesas, reembolso e centro de custo",
              "seguranca": "segurança da informação, senhas, dados confidenciais e incidentes",
              "ti": "computadores, notebooks, VPN, software e suporte técnico",
              "beneficios": "plano de saúde, vale-refeição, cursos e benefícios"}
areas = list(descricoes); E_areas = embed(descricoes.values())

def classificar_area(q):
    return areas[int(np.argmax(E_areas @ embed([q])[0]))]

acerto = np.mean([classificar_area(c["q"]) == c["area"] for c in consultas])
print(f"acurácia do roteador: {acerto:.2f}")

def rank_roteado(q, k=20):
    area = classificar_area(q)
    permitidos = {d["id"] for d in docs if d["area"] == area}
    return [i for i in rank_rerank(q, k=len(docs), n_candidatos=30) if int(i) in permitidos][:k]

print(f"MRR com roteamento: {mrr(rank_roteado, consultas):.3f} | sem: {mrr(rank_rerank, consultas):.3f}")

**Reflexão:** quando o roteador erra a área, o que acontece com a query? Que mecanismo de segurança você colocaria (confidence threshold, busca em duas áreas, fallback sem filtro)?